# Notebook 3 - Parse XML into Holdings Table

this is where we actually read the XML and turn it into a proper table.

each filing has one infoTable element per holding with fields like:
- nameOfIssuer - company name
- cusip - unique identifier for the security (like a stock ticker but more standardized)
- value - dollar value of the position
- sshPrnamt - number of shares
- putCall - if its an option, whether its a call or put

one thing that was annoying - the XML uses namespaces (like ns1:infoTable instead of just infoTable) and the prefix varies between different filers. so you cant just search for "infoTable", you have to strip the namespace prefix first.

also the value field changed units in 2023. before that it was in thousands of dollars, after that its in actual dollars. so old quarters look 1000x smaller if you dont account for this.

In [1]:
import os, sys, pathlib

# ----- COLAB SETUP -----
# Option A: with Google Drive (data stays between sessions)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: without Drive (data gone when session ends, need to rerun)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# detects root automatically when running locally
PROJECT_ROOT = pathlib.Path("/content/13F-Analytics") if pathlib.Path("/content/13F-Analytics").exists() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("working directory:", PROJECT_ROOT)

working directory: /content/13F-Analytics


In [2]:
from src.parser import build_raw_holdings

holdings_raw = build_raw_holdings()
holdings_raw.head(10)

,issuer,title,cusip,value,shares,share_type,put_call,investment_discretion,other_manager,sole,shared,none,value_usd,quarter,report_date,accession_number
0,ALLY FINL INC,COM,02005N100,"498,992,850.00","12,719,675.00",SH,None,DFND,4,"12,719,675.00",0.00,0.00,"498,992,850.00",2026Q1,2026-03-31,0001193125-26-226661
1,ALLY FINL INC,COM,02005N100,"109,996,016.00","2,803,875.00",SH,None,DFND,"2,4,11","2,803,875.00",0.00,0.00,"109,996,016.00",2026Q1,2026-03-31,0001193125-26-226661
2,ALLY FINL INC,COM,02005N100,"165,872,286.00","4,228,200.00",SH,None,DFND,"4,5","4,228,200.00",0.00,0.00,"165,872,286.00",2026Q1,2026-03-31,0001193125-26-226661
3,ALLY FINL INC,COM,02005N100,"123,064,510.00","3,137,000.00",SH,None,DFND,"4,8,11","3,137,000.00",0.00,0.00,"123,064,510.00",2026Q1,2026-03-31,0001193125-26-226661
4,ALLY FINL INC,COM,02005N100,"189,726,088.00","4,836,250.00",SH,None,DFND,"4,10","4,836,250.00",0.00,0.00,"189,726,088.00",2026Q1,2026-03-31,0001193125-26-226661
5,ALLY FINL INC,COM,02005N100,"50,018,250.00","1,275,000.00",SH,None,DFND,"4,11","1,275,000.00",0.00,0.00,"50,018,250.00",2026Q1,2026-03-31,0001193125-26-226661
6,ALPHABET INC,CAP STK CL C,02079K107,"1,028,454,775.00","3,585,215.00",SH,None,DFND,"4,11","3,585,215.00",0.00,0.00,"1,028,454,775.00",2026Q1,2026-03-31,0001193125-26-226661
7,ALPHABET INC,CAP STK CL A,02079K305,"4,802,252.00","16,700.00",SH,None,DFND,"2,4,11","16,700.00",0.00,0.00,"4,802,252.00",2026Q1,2026-03-31,0001193125-26-226661
8,ALPHABET INC,CAP STK CL A,02079K305,"1,926,652,000.00","6,700,000.00",SH,None,DFND,"4,5","6,700,000.00",0.00,0.00,"1,926,652,000.00",2026Q1,2026-03-31,0001193125-26-226661
9,ALPHABET INC,CAP STK CL A,02079K305,"1,797,250,000.00","6,250,000.00",SH,None,DFND,"4,10","6,250,000.00",0.00,0.00,"1,797,250,000.00",2026Q1,2026-03-31,0001193125-26-226661


In [3]:
# how many positions per quarter
holdings_raw.groupby("quarter").agg(
    total_rows=("cusip", "size"),
    unique_issuers=("issuer", "nunique"),
    total_value_usd=("value_usd", "sum"),
).round(0)

,total_rows,unique_issuers,total_value_usd
quarter,,,
2024Q2,129,36,"279,969,062,343.00"
2024Q3,121,37,"266,378,900,503.00"
2024Q4,112,35,"267,175,474,249.00"
2025Q1,4,3,"1,106,550,356.00"
2025Q2,114,37,"257,521,776,925.00"
2025Q3,115,37,"267,334,501,955.00"
2025Q4,110,39,"274,160,086,701.00"
2026Q1,90,26,"263,095,703,570.00"


In [4]:
# check how complete the data is - some fields like put_call are only filled for options
holdings_raw.isna().mean().round(3).sort_values(ascending=False).to_frame("pct_missing")

,pct_missing
put_call,1.00
issuer,0.00
cusip,0.00
title,0.00
value,0.00
shares,0.00
share_type,0.00
investment_discretion,0.00
other_manager,0.00
sole,0.00
